In [69]:
import pandas as pd
import numpy as np
import os
import sys
import time

from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql.functions import lit, col, ceil, sum, avg, udf, coalesce, to_timestamp, date_format
from datetime import datetime, timedelta

In [7]:
java_home = r"C:\Users\n.osipov\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + r"\bin;" + os.environ["PATH"]

python_path = sys.executable
os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [8]:
spark = SparkSession.builder \
    .appName("SkillBox") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

In [9]:
df_january = spark.read.csv(
    "Sales_January_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_february = spark.read.csv(
    "Sales_February_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_march = spark.read.csv(
    "Sales_March_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_april = spark.read.csv(
    "Sales_April_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_may = spark.read.csv(
    "Sales_May_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_june = spark.read.csv(
    "Sales_June_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_july = spark.read.csv(
    "Sales_July_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_august = spark.read.csv(
    "Sales_August_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_september = spark.read.csv(
    "Sales_September_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_october = spark.read.csv(
    "Sales_October_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_november = spark.read.csv(
    "Sales_November_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_december = spark.read.csv(
    "Sales_December_2019.csv",
    header=True,
    inferSchema=True,
    sep=","
)

In [12]:
dfs = [
    df_january, df_february, df_march, df_april,
    df_may, df_june, df_july, df_august,
    df_september, df_october, df_november, df_december
]

In [15]:
df_sales = reduce(lambda a, b: a.union(b), dfs)

In [22]:
df_sales.count()

186850

In [27]:
df_sales = df_sales.drop_duplicates()
df_sales = df_sales.na.drop()
print("Всего:", df_sales.count())
df_sales.show()

Всего: 185686
+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  141395|AAA Batteries (4-...|               1|      2.99|01/29/19 16:20|790 Hickory St, P...|
|  141408|Apple Airpods Hea...|               1|     150.0|01/07/19 13:25|103 14th St, San ...|
|  141798|        20in Monitor|               1|    109.99|01/18/19 07:48|482 Washington St...|
|  142032|       Flatscreen TV|               1|     300.0|01/14/19 09:14|173 Jackson St, S...|
|  142062|    Wired Headphones|               1|     11.99|01/24/19 12:31|930 Church St, Sa...|
|  142211|    Wired Headphones|               1|     11.99|01/03/19 16:01|663 River St, New...|
|  142344|27in 4K Gaming Mo...|               1|    389.99|01/02/19 11:01|336 2nd St, Los A...|
|  142517|Lightning Chargi

In [30]:
df_sales = df_sales.withColumn(
    "Order Date", 
    to_timestamp(col("Order Date"), "MM/dd/yy HH:mm")
).withColumn(
    "Month", 
    date_format(col("Order Date"), "MMMM")
)

In [48]:
# df_sales.orderBy(col("Order ID").desc()).show(10)

In [53]:
df_sales_top10 = (df_sales
    .groupBy("Product")
    .agg(sum("Quantity Ordered").alias("Total Quantity"))
    .orderBy(col("Total Quantity").desc())
    .limit(10)
)

df_sales_top10.show()

+--------------------+--------------+
|             Product|Total Quantity|
+--------------------+--------------+
|AAA Batteries (4-...|         30986|
|AA Batteries (4-p...|         27615|
|USB-C Charging Cable|         23931|
|Lightning Chargin...|         23169|
|    Wired Headphones|         20524|
|Apple Airpods Hea...|         15637|
|Bose SoundSport H...|         13430|
|    27in FHD Monitor|          7541|
|              iPhone|          6847|
|27in 4K Gaming Mo...|          6239|
+--------------------+--------------+



In [62]:
print("=== Без партиционирования ===")
%timeit -n 2 -r 2 df_sales.groupBy("Product").agg(sum("Quantity Ordered")).count()

print("\n=== Имитация партиционированных данных (через фильтр) ===")
%timeit -n 2 -r 2 df_sales.filter(col("Product") == "iPhone").groupBy("Product").agg(sum("Quantity Ordered")).count()

=== Без партиционирования ===
981 ms ± 200 ms per loop (mean ± std. dev. of 2 runs, 2 loops each)

=== Имитация партиционированных данных (через фильтр) ===
676 ms ± 70.3 ms per loop (mean ± std. dev. of 2 runs, 2 loops each)


In [63]:
df_one = df_sales.coalesce(1)
print("Количество партиций:", df_one.rdd.getNumPartitions())

Количество партиций: 1


In [64]:
pdf = df_sales.toPandas()

C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [65]:
os.makedirs("format_comparison", exist_ok=True)

In [66]:
results = []

In [70]:
start = time.time()
pdf.to_csv("format_comparison/data.csv", index=False)
csv_time = time.time() - start
csv_size = os.path.getsize("format_comparison/data.csv") / 1024 / 1024
results.append(("CSV", csv_time, csv_size))

In [71]:
start = time.time()
pdf.to_json("format_comparison/data.json", orient="records", lines=True)
json_time = time.time() - start
json_size = os.path.getsize("format_comparison/data.json") / 1024 / 1024
results.append(("JSON", json_time, json_size))

C:\Users\n.osipov\AppData\Local\Temp\ipykernel_40084\3031618364.py:2: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  pdf.to_json("format_comparison/data.json", orient="records", lines=True)


In [72]:
start = time.time()
pdf.to_parquet("format_comparison/data.parquet", index=False)
parquet_time = time.time() - start
parquet_size = os.path.getsize("format_comparison/data.parquet") / 1024 / 1024
results.append(("Parquet", parquet_time, parquet_size))

In [73]:
print(f"{'Формат':<12} {'Время (сек)':<12} {'Размер (МБ)':<12}")
print("-" * 40)
for fmt, t, s in results:
    print(f"{fmt:<12} {t:<12.3f} {s:<12.2f}")

Формат       Время (сек)  Размер (МБ) 
----------------------------------------
CSV          0.717        17.84       
JSON         0.502        34.13       
Parquet      0.130        4.54        


In [81]:
def parse_address(address):
    try:
        if address is None:
            return (None, None, None, None)
        
        parts = address.split(",")
        
        street = parts[0].strip() if len(parts) > 0 else None
        city = parts[1].strip() if len(parts) > 1 else None
        
        state_zip = parts[2].strip().split() if len(parts) > 2 else []
        state = state_zip[0] if len(state_zip) > 0 else None
        postal = state_zip[1] if len(state_zip) > 1 else None
        
        return (street, city, state, postal)
    except:
        return (None, None, None, None)

In [82]:
address_schema = StructType([
    StructField("Street", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Postal Code", StringType(), True)
])

In [83]:
parse_address_udf = udf(parse_address, address_schema)

In [84]:
df_parsed = df_sales.withColumn("Address Parsed", parse_address_udf(col("Purchase Address")))

In [85]:
df_parsed = df_parsed \
    .withColumn("Street", col("Address Parsed.Street")) \
    .withColumn("City", col("Address Parsed.City")) \
    .withColumn("State", col("Address Parsed.State")) \
    .withColumn("Postal Code", col("Address Parsed.Postal Code")) \
    .drop("Address Parsed")

In [86]:
df_parsed.select("Purchase Address", "Street", "City", "State", "Postal Code").show(5, truncate=False)

+---------------------------------------+-----------------+-------------+-----+-----------+
|Purchase Address                       |Street           |City         |State|Postal Code|
+---------------------------------------+-----------------+-------------+-----+-----------+
|790 Hickory St, Portland, ME 04101     |790 Hickory St   |Portland     |ME   |04101      |
|103 14th St, San Francisco, CA 94016   |103 14th St      |San Francisco|CA   |94016      |
|482 Washington St, Boston, MA 02215    |482 Washington St|Boston       |MA   |02215      |
|173 Jackson St, San Francisco, CA 94016|173 Jackson St   |San Francisco|CA   |94016      |
|930 Church St, San Francisco, CA 94016 |930 Church St    |San Francisco|CA   |94016      |
+---------------------------------------+-----------------+-------------+-----+-----------+
only showing top 5 rows


In [87]:
df_cached = df_parsed.cache()

In [88]:
df_final = df_cached.withColumn(
    "Total Price", 
    col("Quantity Ordered") * col("Price Each")
)

In [89]:
df_final.select("Product", "Quantity Ordered", "Price Each", "Total Price").show(5)

+--------------------+----------------+----------+-----------+
|             Product|Quantity Ordered|Price Each|Total Price|
+--------------------+----------------+----------+-----------+
|AAA Batteries (4-...|               1|      2.99|       2.99|
|Apple Airpods Hea...|               1|     150.0|      150.0|
|        20in Monitor|               1|    109.99|     109.99|
|       Flatscreen TV|               1|     300.0|      300.0|
|    Wired Headphones|               1|     11.99|      11.99|
+--------------------+----------------+----------+-----------+
only showing top 5 rows


In [90]:
start = time.time()
(df_sales
    .withColumn("Address Parsed", parse_address_udf(col("Purchase Address")))
    .withColumn("Total Price", col("Quantity Ordered") * col("Price Each"))
    .count()
)
print(f"Без кеширования: {time.time() - start:.2f} сек")

Без кеширования: 4.28 сек


In [92]:
start = time.time()
df_cached = (df_sales
    .withColumn("Address Parsed", parse_address_udf(col("Purchase Address")))
    .cache()
)
df_cached.count()

(df_cached
    .withColumn("Total Price", col("Quantity Ordered") * col("Price Each"))
    .count()
)
print(f"С кешированием: {time.time() - start:.2f} сек")

С кешированием: 3.40 сек
